# Subject01 flatmap -> MindEye2 SDXL-unCLIP inference

Concise Brain-IT-style inference: load the trained low-level SD-VAE bridge, load a semantic CLIP-token bridge, predict both from flatmap brain tokens, initialize MindEye2 SDXL-unCLIP from the low-level image, condition on predicted OpenCLIP tokens, then evaluate the shared-1000 reconstructions.


## 0. Setup + paths


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive mount skipped/failed: {type(e).__name__}: {e}")

import os, re, glob, json, random, shutil, time, gc, sys, subprocess
from collections import defaultdict, OrderedDict

_missing = []
for module_name, package_name in [
    ("diffusers", "diffusers"),
    ("accelerate", "accelerate"),
    ("transformers", "transformers"),
    ("safetensors", "safetensors"),
    ("skimage", "scikit-image"),
    ("omegaconf", "omegaconf"),
    ("pytorch_lightning", "pytorch-lightning"),
    ("einops", "einops"),
    ("open_clip", "open_clip_torch"),
    ("kornia", "kornia"),
    ("webdataset", "webdataset"),
]
for module_name, package_name in list(_missing):
    pass
_missing = [pkg for mod, pkg in [
    ("diffusers", "diffusers"), ("accelerate", "accelerate"), ("transformers", "transformers"),
    ("safetensors", "safetensors"), ("skimage", "scikit-image"), ("omegaconf", "omegaconf"),
    ("pytorch_lightning", "pytorch-lightning"), ("einops", "einops"), ("open_clip", "open_clip_torch"),
    ("kornia", "kornia"), ("webdataset", "webdataset")
] if __import__('importlib').util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.io import loadmat
from skimage.color import rgb2gray
from skimage.metrics import structural_similarity as ssim_fn
from diffusers import AutoencoderKL
try:
    from diffusers.models.vae import Decoder
except Exception:
    from diffusers.models.autoencoders.vae import Decoder

BASE = "/content"
DRIVE_PROJECT_DIR = f"{BASE}/drive/Shareddrives/FMRI_Paper"
DRIVE_INPUT_DIR = f"{DRIVE_PROJECT_DIR}/inputs"
LOCAL_INPUT_DIR = f"{BASE}/inputs"
GCS_INPUT_ROOT = "gs://colab-bucket-maxwell/inputs"
USE_GCS_INPUTS = True

SUBJECT = "subj01"
TRIALS_PER_SESSION = 750
KNOWN_BAD_SESSIONS = {("subj01", 11)}

# Low-level checkpoint produced by subject01_visual_brain_sdvae_transformers*.ipynb.
LOWLEVEL_CKPT_PATH = f"{DRIVE_PROJECT_DIR}/outputs/subject01_flatmap_mindeye_sd_bridge_20260831_061019/best_mindeye_sd_flatmap_bridge.pt"

# Semantic bridge checkpoint: must output MindEye2-compatible OpenCLIP ViT-bigG/14 tokens [B, 256, 1664].
TLDR_PATH = ""  # set this to your trained fMRI->OpenCLIP-token transformer checkpoint
CLIP_BRIDGE_CKPT_PATH = TLDR_PATH

# MindEye2 / Brain-IT diffusion assets.
BRAINIT_DIR = f"{BASE}/brainit-fmri"
MINDEYE2_UNCLIP_CKPT_PATH = f"{DRIVE_INPUT_DIR}/unclip6_epoch0_step110000.ckpt"
MINDEYE2_UNCLIP_CONFIG = f"{BRAINIT_DIR}/src/MindEyeV2/generative_models/configs/unclip6.yaml"

VAE_CKPT_URL = "https://huggingface.co/datasets/pscotti/mindeyev2/resolve/main/sd_image_var_autoenc.pth"
VAE_CKPT_PATH = f"{DRIVE_INPUT_DIR}/sd_image_var_autoenc.pth"
VAE_IMAGE_SIZE = 512
VAE_SCALING_FACTOR = 0.18215
VAE_DECODE_BATCH = 16

MINDEYE_SD_CACHE_DIR = f"{DRIVE_INPUT_DIR}/subject01_mindeye_sd_image_var_autoenc_features_{VAE_IMAGE_SIZE}"
MINDEYE_SD_LOCAL_CACHE_DIR = f"{LOCAL_INPUT_DIR}/subject01_mindeye_sd_image_var_autoenc_features_{VAE_IMAGE_SIZE}"
FLATMAP_DRIVE_DIR = f"{DRIVE_INPUT_DIR}/flatmap_brain_tokens"
FLATMAP_LOCAL_DIR = f"{LOCAL_INPUT_DIR}/flatmap_brain_tokens"

RUN_TIMESTAMP = ""  # set old timestamp to resume
if not RUN_TIMESTAMP:
    RUN_TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/outputs/subject01_mindeye2_unclip_inference_{RUN_TIMESTAMP}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
FINAL_N = 1000
CLIP_NUM_TOKENS = 256
CLIP_DIM = 1664
LOWLEVEL_SIDE = 16
LOWLEVEL_CHANNELS = 64
D_MODEL = 768
N_LAYERS = 4
N_HEADS = 8
FFN_DIM = 3072
DROPOUT = 0.10
DIFFUSION_START_STEP = 14
DIFFUSION_NUM_STEPS = 38
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass
print(f"device={DEVICE} amp={USE_AMP} output={OUTPUT_DIR}")


## 1. Hydrate small inputs + index shared-1000


In [ ]:
def gcloud_cp(src, dst, recursive=False):
    cmd = ["gcloud", "storage", "cp"]
    if recursive:
        cmd.append("--recursive")
    cmd += [src, dst]
    subprocess.check_call(cmd)

os.makedirs(LOCAL_INPUT_DIR, exist_ok=True)
os.makedirs(FLATMAP_LOCAL_DIR, exist_ok=True)

EXP_LOCAL = f"{LOCAL_INPUT_DIR}/nsd_expdesign.mat"
EXP_DRIVE = f"{DRIVE_INPUT_DIR}/nsd_expdesign.mat"
if not os.path.exists(EXP_LOCAL):
    if os.path.exists(EXP_DRIVE):
        shutil.copy2(EXP_DRIVE, EXP_LOCAL)
    elif USE_GCS_INPUTS:
        gcloud_cp(f"{GCS_INPUT_ROOT}/nsd_expdesign.mat", EXP_LOCAL)
    else:
        raise FileNotFoundError("Missing nsd_expdesign.mat")
mat = loadmat(EXP_LOCAL)

if USE_GCS_INPUTS and not os.path.exists(f"{MINDEYE_SD_LOCAL_CACHE_DIR}/metadata.json"):
    gcloud_cp(f"{GCS_INPUT_ROOT}/subject01_mindeye_sd_image_var_autoenc_features_{VAE_IMAGE_SIZE}", LOCAL_INPUT_DIR, recursive=True)

masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1
subjectim = mat["subjectim"].astype(np.int64) - 1
subject_idx = int(SUBJECT[-2:]) - 1
imgbrick_ids_all = subjectim[subject_idx, masterordering]
shared_ids = set(mat["sharedix"].reshape(-1).astype(np.int64) - 1)

session_nums = list(range(1, 41))
session_nums = [s for s in session_nums if (SUBJECT, s) not in KNOWN_BAD_SESSIONS]
print(f"sessions={session_nums}")

def ids_for_session(sess, n_trials=TRIALS_PER_SESSION):
    start = (sess - 1) * TRIALS_PER_SESSION
    return imgbrick_ids_all[start:start + n_trials].astype(int)

def flatmap_local_path(sess):
    return f"{FLATMAP_LOCAL_DIR}/{SUBJECT}_flatmap_tokens_session{sess:02d}.pt"

def flatmap_drive_path(sess):
    return f"{FLATMAP_DRIVE_DIR}/{SUBJECT}_flatmap_tokens_session{sess:02d}.pt"

def ensure_flatmap_session(sess):
    local = flatmap_local_path(sess)
    if os.path.exists(local):
        return local
    drive = flatmap_drive_path(sess)
    if os.path.exists(drive):
        shutil.copy2(drive, local)
        return local
    if USE_GCS_INPUTS:
        gcloud_cp(f"{GCS_INPUT_ROOT}/flatmap_brain_tokens/{SUBJECT}_flatmap_tokens_session{sess:02d}.pt", local)
        return local
    raise FileNotFoundError(local)


## 2. Load frozen MindEye SD autoencoder + target cache


In [ ]:
def download_with_progress(url, dst, desc="download"):
    import urllib.request
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = f"{dst}.part"
    if os.path.exists(tmp):
        os.remove(tmp)
    with urllib.request.urlopen(url) as response, open(tmp, "wb") as f:
        total = int(response.headers.get("Content-Length", 0))
        with tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                f.write(chunk)
                pbar.update(len(chunk))
    os.replace(tmp, dst)

if not os.path.exists(VAE_CKPT_PATH):
    download_with_progress(VAE_CKPT_URL, VAE_CKPT_PATH, "download sd_image_var_autoenc.pth")

vae = AutoencoderKL(
    down_block_types=["DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D", "DownEncoderBlock2D"],
    up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D"],
    block_out_channels=[128, 256, 512, 512],
    layers_per_block=2,
    sample_size=256,
)
vae.load_state_dict(torch.load(VAE_CKPT_PATH, map_location="cpu"))
vae = vae.to(DEVICE).eval().requires_grad_(False)
print(f"loaded MindEye SD autoencoder: {VAE_CKPT_PATH}")

metadata_path = f"{MINDEYE_SD_LOCAL_CACHE_DIR}/metadata.json" if os.path.exists(f"{MINDEYE_SD_LOCAL_CACHE_DIR}/metadata.json") else f"{MINDEYE_SD_CACHE_DIR}/metadata.json"
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Missing MindEye SD cache metadata: {metadata_path}")
with open(metadata_path) as f:
    mindeye_sd_manifest = json.load(f)
MINDEYE_SD_READ_CACHE_DIR = os.path.dirname(metadata_path)
LATENT_SHAPE = tuple(mindeye_sd_manifest["latent_shape"])
assert LATENT_SHAPE == (4, 64, 64), LATENT_SHAPE

_mindeye_sd_lookup = {}
for shard in mindeye_sd_manifest["shards"]:
    shard_path = f"{MINDEYE_SD_READ_CACHE_DIR}/{shard['file']}"
    for row, img_id in enumerate(shard["img_ids"]):
        _mindeye_sd_lookup[int(img_id)] = (shard_path, row)
_mindeye_sd_lru = OrderedDict()

def _load_mindeye_sd_shard(path):
    if path in _mindeye_sd_lru:
        _mindeye_sd_lru.move_to_end(path)
        return _mindeye_sd_lru[path]
    shard = torch.load(path, map_location="cpu")
    _mindeye_sd_lru[path] = shard
    while len(_mindeye_sd_lru) > 32:
        _mindeye_sd_lru.popitem(last=False)
    return shard

def mindeye_sd_targets_for_ids(img_ids, include_images=False, device=DEVICE):
    ids = [int(i) for i in torch.as_tensor(img_ids).detach().cpu().tolist()]
    latents = torch.empty((len(ids), *LATENT_SHAPE), dtype=torch.float16)
    images = torch.empty((len(ids), 3, VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), dtype=torch.uint8) if include_images else None
    by_shard = defaultdict(list)
    for out_row, img_id in enumerate(ids):
        by_shard[_mindeye_sd_lookup[img_id][0]].append((out_row, _mindeye_sd_lookup[img_id][1]))
    for shard_path, pairs in by_shard.items():
        shard = _load_mindeye_sd_shard(shard_path)
        out_rows = [p[0] for p in pairs]
        shard_rows = torch.as_tensor([p[1] for p in pairs], dtype=torch.long)
        latents[out_rows] = shard["latents"][shard_rows]
        if include_images:
            images[out_rows] = shard["images_u8"][shard_rows]
    out = {"latents": latents.float().to(device, non_blocking=True) if device is not None else latents.float()}
    if include_images:
        out["images"] = images.float().div(255.0)
    return out

@torch.no_grad()
def encode_mindeye_sd_images(img01):
    img_m11 = img01.to(DEVICE, non_blocking=True).mul(2).sub(1)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        return (vae.encode(img_m11).latent_dist.mode() * VAE_SCALING_FACTOR).float().cpu()

@torch.no_grad()
def decode_mindeye_sd_latents(latents):
    latents = latents.to(DEVICE, non_blocking=True) / VAE_SCALING_FACTOR
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        imgs = vae.decode(latents).sample
    return imgs.float().add(1).div(2).clamp(0, 1).cpu()


## 3. Models: low-level bridge + semantic CLIP bridge


In [ ]:
class CrossAttentionBridge(nn.Module):
    def __init__(self, brain_dim, target_dim, num_queries, d_model, n_layers, n_heads, ffn_dim, dropout):
        super().__init__()
        self.query_embed = nn.Parameter(torch.randn(num_queries, d_model) * 0.02)
        self.brain_proj = nn.Linear(brain_dim, d_model) if brain_dim != d_model else nn.Identity()
        self.brain_norm = nn.LayerNorm(d_model)
        layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=ffn_dim, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, target_dim)
    def forward(self, brain_tokens):
        b = brain_tokens.shape[0]
        memory = self.brain_norm(self.brain_proj(brain_tokens))
        queries = self.query_embed.unsqueeze(0).repeat(b, 1, 1)
        return self.output_proj(self.decoder(tgt=queries, memory=memory))

class FlatmapMindEyeSDBridge(nn.Module):
    def __init__(self, brain_dim, d_model, n_layers, n_heads, ffn_dim, dropout):
        super().__init__()
        self.bridge = CrossAttentionBridge(brain_dim, LOWLEVEL_CHANNELS, LOWLEVEL_SIDE * LOWLEVEL_SIDE, d_model, n_layers, n_heads, ffn_dim, dropout)
        self.norm = nn.GroupNorm(1, LOWLEVEL_CHANNELS)
        self.upsampler = Decoder(
            in_channels=LOWLEVEL_CHANNELS,
            out_channels=LATENT_SHAPE[0],
            up_block_types=["UpDecoderBlock2D", "UpDecoderBlock2D", "UpDecoderBlock2D"],
            block_out_channels=[64, 128, 256],
            layers_per_block=1,
        )
    def forward(self, brain_tokens):
        seed_tokens = self.bridge(brain_tokens)
        seed = seed_tokens.transpose(1, 2).reshape(seed_tokens.shape[0], LOWLEVEL_CHANNELS, LOWLEVEL_SIDE, LOWLEVEL_SIDE).contiguous()
        return self.upsampler(self.norm(seed))

class FlatmapCLIPBridge(nn.Module):
    def __init__(self, brain_dim, clip_dim=CLIP_DIM, num_queries=CLIP_NUM_TOKENS, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT):
        super().__init__()
        self.bridge = CrossAttentionBridge(brain_dim, clip_dim, num_queries, d_model, n_layers, n_heads, ffn_dim, dropout)
    def forward(self, brain_tokens):
        return self.bridge(brain_tokens)

def load_ckpt_payload(path):
    if not path or not os.path.exists(path):
        raise FileNotFoundError(path or "empty checkpoint path")
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict):
        return obj
    raise TypeError(f"Expected checkpoint dict, got {type(obj)}")

low_ckpt = load_ckpt_payload(LOWLEVEL_CKPT_PATH)
brain_dim = int(low_ckpt.get("brain_dim", 768))
lowlevel_model = FlatmapMindEyeSDBridge(
    brain_dim=brain_dim,
    d_model=int(low_ckpt.get("d_model", D_MODEL)),
    n_layers=int(low_ckpt.get("n_layers", N_LAYERS)),
    n_heads=int(low_ckpt.get("n_heads", N_HEADS)),
    ffn_dim=int(low_ckpt.get("ffn_dim", FFN_DIM)),
    dropout=float(low_ckpt.get("dropout", DROPOUT)),
).to(DEVICE).eval()
lowlevel_model.load_state_dict(low_ckpt["model"])
lowlevel_model.requires_grad_(False)
print(f"loaded low-level bridge: {LOWLEVEL_CKPT_PATH}")

clip_ckpt = load_ckpt_payload(CLIP_BRIDGE_CKPT_PATH)
clip_brain_dim = int(clip_ckpt.get("brain_dim", brain_dim))
clip_dim = int(clip_ckpt.get("clip_dim", clip_ckpt.get("target_dim", CLIP_DIM)))
clip_queries = int(clip_ckpt.get("num_queries", CLIP_NUM_TOKENS))
if (clip_queries, clip_dim) != (CLIP_NUM_TOKENS, CLIP_DIM):
    raise ValueError(f"MindEye2 unCLIP expects [B,{CLIP_NUM_TOKENS},{CLIP_DIM}], checkpoint gives [B,{clip_queries},{clip_dim}].")
clip_model = FlatmapCLIPBridge(
    brain_dim=clip_brain_dim,
    clip_dim=clip_dim,
    num_queries=clip_queries,
    d_model=int(clip_ckpt.get("d_model", D_MODEL)),
    n_layers=int(clip_ckpt.get("n_layers", N_LAYERS)),
    n_heads=int(clip_ckpt.get("n_heads", N_HEADS)),
    ffn_dim=int(clip_ckpt.get("ffn_dim", FFN_DIM)),
    dropout=float(clip_ckpt.get("dropout", DROPOUT)),
).to(DEVICE).eval()
clip_model.load_state_dict(clip_ckpt["model"])
clip_model.requires_grad_(False)
print(f"loaded semantic CLIP bridge: {CLIP_BRIDGE_CKPT_PATH}")


## 4. Shared-1000 flatmap tokens


In [ ]:
SHARED_TOKENS_PATH = f"{OUTPUT_DIR}/shared1000_flatmap_tokens.pt"
IDS_PATH = f"{OUTPUT_DIR}/shared1000_img_ids.json"
SESSION_PART_DIR = f"{OUTPUT_DIR}/shared1000_session_parts"
os.makedirs(SESSION_PART_DIR, exist_ok=True)

def load_flatmap_tokens(sess):
    path = ensure_flatmap_session(sess)
    return torch.load(path, map_location="cpu").float()

def session_part_path(sess):
    return f"{SESSION_PART_DIR}/session{sess:02d}.pt"

def collect_shared1000_flatmap_tokens():
    if os.path.exists(SHARED_TOKENS_PATH) and os.path.exists(IDS_PATH):
        with open(IDS_PATH) as f:
            ids = json.load(f)
        return torch.load(SHARED_TOKENS_PATH, map_location="cpu"), ids
    shared_id_set = set(int(i) for i in shared_ids)
    for sess in tqdm(session_nums, desc="collect shared-1000 session parts"):
        part_path = session_part_path(sess)
        if os.path.exists(part_path):
            continue
        brain_tokens = load_flatmap_tokens(sess)
        sess_ids = ids_for_session(sess, brain_tokens.shape[0])
        rows = [r for r, img_id in enumerate(sess_ids) if int(img_id) in shared_id_set]
        part = {"session": int(sess), "img_ids": [int(sess_ids[r]) for r in rows], "brain_tokens": brain_tokens[rows].half().cpu()}
        torch.save(part, f"{part_path}.tmp")
        os.replace(f"{part_path}.tmp", part_path)
        del brain_tokens, part
        gc.collect()
    groups = defaultdict(list)
    for sess in session_nums:
        part = torch.load(session_part_path(sess), map_location="cpu")
        for img_id, token in zip(part["img_ids"], part["brain_tokens"]):
            groups[int(img_id)].append(token)
    ids = sorted(groups)[:FINAL_N]
    avg_tokens = torch.stack([torch.stack(groups[i]).mean(0) for i in ids]).half().contiguous()
    torch.save(avg_tokens, f"{SHARED_TOKENS_PATH}.tmp")
    os.replace(f"{SHARED_TOKENS_PATH}.tmp", SHARED_TOKENS_PATH)
    with open(f"{IDS_PATH}.tmp", "w") as f:
        json.dump([int(i) for i in ids], f)
    os.replace(f"{IDS_PATH}.tmp", IDS_PATH)
    return avg_tokens, ids

shared_brain_tokens, shared_img_ids = collect_shared1000_flatmap_tokens()
print(tuple(shared_brain_tokens.shape), len(shared_img_ids))


## 5. Predict low-level init images + OpenCLIP tokens


In [ ]:
LOWLEVEL_RECONS_PATH = f"{OUTPUT_DIR}/shared1000_lowlevel_uint8.pt"
CLIP_TOKENS_PATH = f"{OUTPUT_DIR}/shared1000_openclip_bigG_tokens_fp16.pt"

@torch.no_grad()
def predict_lowlevel_uint8(brain_tokens):
    if os.path.exists(LOWLEVEL_RECONS_PATH):
        return torch.load(LOWLEVEL_RECONS_PATH, map_location="cpu")
    recons = []
    for start in tqdm(range(0, len(brain_tokens), BATCH_SIZE), desc="predict low-level SD-VAE"):
        xb = brain_tokens[start:start+BATCH_SIZE].to(DEVICE, non_blocking=True).float()
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            latents = lowlevel_model(xb).float()
        imgs = decode_mindeye_sd_latents(latents)
        recons.append(imgs.mul(255).round().byte())
    recons = torch.cat(recons, dim=0)
    torch.save(recons, f"{LOWLEVEL_RECONS_PATH}.tmp")
    os.replace(f"{LOWLEVEL_RECONS_PATH}.tmp", LOWLEVEL_RECONS_PATH)
    return recons

@torch.no_grad()
def predict_clip_tokens(brain_tokens):
    if os.path.exists(CLIP_TOKENS_PATH):
        return torch.load(CLIP_TOKENS_PATH, map_location="cpu")
    preds = []
    for start in tqdm(range(0, len(brain_tokens), BATCH_SIZE), desc="predict OpenCLIP tokens"):
        xb = brain_tokens[start:start+BATCH_SIZE].to(DEVICE, non_blocking=True).float()
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = clip_model(xb).float()
        if pred.shape[1:] != (CLIP_NUM_TOKENS, CLIP_DIM):
            raise ValueError(f"bad clip token shape {tuple(pred.shape)}")
        preds.append(pred.half().cpu())
    preds = torch.cat(preds, dim=0).contiguous()
    torch.save(preds, f"{CLIP_TOKENS_PATH}.tmp")
    os.replace(f"{CLIP_TOKENS_PATH}.tmp", CLIP_TOKENS_PATH)
    return preds

lowlevel_uint8 = predict_lowlevel_uint8(shared_brain_tokens)
openclip_tokens = predict_clip_tokens(shared_brain_tokens)
print("lowlevel", tuple(lowlevel_uint8.shape), lowlevel_uint8.dtype)
print("openclip", tuple(openclip_tokens.shape), openclip_tokens.dtype)


## 6. Load MindEye2 SDXL-unCLIP and generate final images


In [ ]:
if not os.path.exists(BRAINIT_DIR):
    subprocess.check_call(["git", "clone", "https://github.com/WeizmannVision/brainit-fmri.git", BRAINIT_DIR])
sys.path.insert(0, BRAINIT_DIR)
sys.path.insert(0, f"{BRAINIT_DIR}/src/MindEyeV2")
sys.path.insert(0, f"{BRAINIT_DIR}/src/MindEyeV2/generative_models")

if not os.path.exists(MINDEYE2_UNCLIP_CKPT_PATH):
    raise FileNotFoundError(f"Missing MindEye2 unCLIP checkpoint: {MINDEYE2_UNCLIP_CKPT_PATH}")

from utils.diffusion_utils import load_diffusion_engine
from src.MindEyeV2.utils import unclip_recon_new

diffusion_engine = load_diffusion_engine(config_path=MINDEYE2_UNCLIP_CONFIG, ckpt_path=MINDEYE2_UNCLIP_CKPT_PATH)
diffusion_engine = diffusion_engine.to(DEVICE).eval().requires_grad_(False)
diffusion_engine.sampler.num_steps = DIFFUSION_NUM_STEPS

@torch.no_grad()
def get_vector_suffix():
    batch = {
        "jpg": torch.randn(1, 3, 1, 1, device=DEVICE),
        "original_size_as_tuple": torch.ones(1, 2, device=DEVICE) * 256,
        "crop_coords_top_left": torch.zeros(1, 2, device=DEVICE),
    }
    return diffusion_engine.conditioner(batch)["vector"]

vector_suffix = get_vector_suffix()
print("vector_suffix", tuple(vector_suffix.shape))


In [ ]:
FINAL_RECON_DIR = f"{OUTPUT_DIR}/shared1000_mindeye2_unclip_final_uint8"
FINAL_RECONS_PATH = f"{OUTPUT_DIR}/shared1000_mindeye2_unclip_final_uint8.pt"
os.makedirs(FINAL_RECON_DIR, exist_ok=True)

def final_part_path(i):
    return f"{FINAL_RECON_DIR}/img_{i:04d}.pt"

@torch.no_grad()
def generate_one_final(i):
    low_img = lowlevel_uint8[i:i+1].float().div(255.0)
    low_img_256 = F.interpolate(low_img, size=(256, 256), mode="bilinear", align_corners=False).to(DEVICE)
    init_latent = diffusion_engine.encode_first_stage(low_img_256 * 2 - 1)
    clip = openclip_tokens[i:i+1].to(DEVICE).float()
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        sample = unclip_recon_new(
            clip,
            diffusion_engine,
            vector_suffix=vector_suffix,
            num_samples=1,
            init_image_latent=init_latent,
            start_step=DIFFUSION_START_STEP,
        )
    sample = sample[0].float().clamp(0, 1).cpu()
    return sample.mul(255).round().byte()

if not os.path.exists(FINAL_RECONS_PATH):
    for i in tqdm(range(min(FINAL_N, len(openclip_tokens))), desc="MindEye2 SDXL-unCLIP"):
        path = final_part_path(i)
        if os.path.exists(path):
            continue
        img = generate_one_final(i)
        torch.save(img, f"{path}.tmp")
        os.replace(f"{path}.tmp", path)
        if i % 25 == 0:
            torch.cuda.empty_cache()
    final = torch.stack([torch.load(final_part_path(i), map_location="cpu") for i in range(min(FINAL_N, len(openclip_tokens)))])
    torch.save(final, f"{FINAL_RECONS_PATH}.tmp")
    os.replace(f"{FINAL_RECONS_PATH}.tmp", FINAL_RECONS_PATH)
else:
    final = torch.load(FINAL_RECONS_PATH, map_location="cpu")
print("final", tuple(final.shape), final.dtype)


## 7. Shared-1000 eval


In [ ]:
FINAL_EVAL_PATH = f"{OUTPUT_DIR}/shared1000_mindeye2_unclip_final_eval.csv"

if os.path.exists(FINAL_EVAL_PATH):
    metrics_df = pd.read_csv(FINAL_EVAL_PATH, index_col=0)
    display(metrics_df)
else:
    final_uint8 = torch.load(FINAL_RECONS_PATH, map_location="cpu")
    recons = final_uint8.float().div(255.0)
    if recons.shape[-1] != VAE_IMAGE_SIZE:
        recons = F.interpolate(recons, size=(VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
    img_ids = torch.tensor(shared_img_ids[:len(recons)], dtype=torch.long)

    pixel_mse_total = 0.0
    pixcorr_scores, ssim_scores = [], []
    latent_mse_total, latent_cos_total, seen = 0.0, 0.0, 0
    for start in tqdm(range(0, len(recons), 32), desc="eval final recons"):
        rec = recons[start:start+32].float()
        batch_ids = img_ids[start:start+32]
        cache = mindeye_sd_targets_for_ids(batch_ids, include_images=True, device=None)
        true_img = cache["images"].float()
        n = len(rec)
        pixel_mse_total += n * float(F.mse_loss(rec, true_img).cpu())
        r = rec.flatten(1).numpy(); t = true_img.flatten(1).numpy()
        pixcorr_scores.extend(float(np.corrcoef(t[i], r[i])[0, 1]) for i in range(n))
        rec_gray = rgb2gray(rec.permute(0, 2, 3, 1).numpy())
        true_gray = rgb2gray(true_img.permute(0, 2, 3, 1).numpy())
        ssim_scores.extend(ssim_fn(rec_gray[i], true_gray[i], data_range=1.0, gaussian_weights=True, sigma=1.5, use_sample_covariance=False) for i in range(n))
        rec_latents = encode_mindeye_sd_images(rec)
        true_latents = cache["latents"].float()
        latent_mse_total += n * float(F.mse_loss(rec_latents, true_latents).cpu())
        latent_cos_total += n * float(F.cosine_similarity(rec_latents.flatten(1), true_latents.flatten(1), dim=1).mean().cpu())
        seen += n
    metrics = {
        "N": seen,
        "Pixel_MSE": pixel_mse_total / seen,
        "PixCorr": float(np.mean(pixcorr_scores)),
        "SSIM": float(np.mean(ssim_scores)),
        "MindEyeSD_Latent_MSE": latent_mse_total / seen,
        "MindEyeSD_Latent_Cosine": latent_cos_total / seen,
    }
    metrics_df = pd.DataFrame([metrics]).set_index("N")
    display(metrics_df)
    metrics_df.to_csv(FINAL_EVAL_PATH)
    print(f"saved={FINAL_EVAL_PATH}")


## 8. Preview grid


In [ ]:
PREVIEW_GRID_N = 48
PREVIEW_GRID_COLS = 6
PREVIEW_GRID_PATH = f"{OUTPUT_DIR}/shared1000_mindeye2_unclip_preview_grid.png"

if os.path.exists(PREVIEW_GRID_PATH):
    print(f"preview grid exists: {PREVIEW_GRID_PATH}")
else:
    final_uint8 = torch.load(FINAL_RECONS_PATH, map_location="cpu")
    final_img = final_uint8.float().div(255.0)
    if final_img.shape[-1] != VAE_IMAGE_SIZE:
        final_img = F.interpolate(final_img, size=(VAE_IMAGE_SIZE, VAE_IMAGE_SIZE), mode="bicubic", align_corners=False).clamp(0, 1)
    low_img = torch.load(LOWLEVEL_RECONS_PATH, map_location="cpu").float().div(255.0)
    n_show = min(PREVIEW_GRID_N, len(final_img))
    rng = np.random.RandomState(SEED)
    show_idx = rng.choice(len(final_img), size=n_show, replace=False)
    true_img = mindeye_sd_targets_for_ids(torch.tensor([shared_img_ids[int(i)] for i in show_idx]), include_images=True, device=None)["images"].float()
    combined = torch.cat([true_img, low_img[show_idx], final_img[show_idx]], dim=3).clamp(0, 1)
    cols = min(PREVIEW_GRID_COLS, n_show)
    rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(7.2 * cols, 3.0 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis("off")
    for k, idx in enumerate(show_idx):
        ax = axes[k // cols, k % cols]
        ax.imshow(combined[k].permute(1, 2, 0))
        ax.set_title(f"id {shared_img_ids[int(idx)]}: true | low | final", fontsize=8)
    plt.savefig(PREVIEW_GRID_PATH, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"saved={PREVIEW_GRID_PATH}")
